# Módulo 01 · Aula 6 (Exercício, solução): Primeiro RAG de ponta a ponta

**Duração:** 45 min

## Enunciado

A equipe de atendimento da TechNova quer ver um primeiro assistente que responda dúvidas sobre as **políticas da
loja** com base nos documentos, e não no "conhecimento" do modelo. Você vai montar as duas fases de um RAG sobre as 8
políticas em Markdown de `data/documents/policies/`:

```mermaid
flowchart LR
    subgraph ING[Ingestão · código do produto]
        direction LR
        P[(8 políticas .md)] --> L[load_file] --> C[split_documents] --> I[(build_index<br/>FAISS em memória)]
    end
    subgraph CON[Consulta · código seu]
        direction LR
        Q([pergunta]) --> B[top 3<br/>trechos] --> M[prompt<br/>com contexto] --> G[get_llm] --> R([resposta])
    end
    I --> B
```

A **ingestão** usa o produto que você conheceu nas aulas anteriores: `load_file` e `split_documents`
([src/rag/ingestion.py](../src/rag/ingestion.py)) e `build_index` ([src/rag/vectorstore.py](../src/rag/vectorstore.py)).
A **consulta** é escrita por você: busca, prompt e resposta com `get_llm()`. Não use `src/rag/retrieval.py` nem
`src/rag/generation.py`; eles são o assunto do módulo 02, e este exercício mostra o que eles vão melhorar.

## Atividades

1. Carregar as 8 políticas com `load_file`.
2. Dividir em trechos com `split_documents`.
3. Criar um índice FAISS **em memória** com `build_index`.
4. Buscar os 3 trechos mais próximos de uma pergunta.
5. Montar o prompt com o contexto e responder com `get_llm()`; testar 3 perguntas (troca, frete e uma que a base não
   responde).
6. Comparar com a resposta do modelo **sem** contexto.
7. (Opcional) Desafio: filtrar a política arquivada e ver a diferença na pergunta de frete.

## Critérios de sucesso

- Todas as células de verificação imprimem ✅.
- O índice tem exatamente o número esperado de trechos (88, com `CHUNK_SIZE=1000` e `CHUNK_OVERLAP=150`).
- A busca traz a política certa entre os 3 primeiros trechos, na pergunta de troca e na de frete.
- A resposta sobre troca cita o prazo correto (30 dias).
- A pergunta que a base não responde recebe a frase de "não encontrei", sem nada inventado.

> 💰 **Economia:** as verificações das atividades 1 a 4 não chamam o LLM (a atividade 3 e as buscas usam embeddings,
> bem pouco). Com a solução completa, o notebook faz 6 chamadas ao LLM (7 com o desafio).

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 0. Preparação (fornecido)

Execute a célula abaixo sem alterações. Ela importa o que você vai usar do produto, cria o cliente de chat e define as
perguntas de teste e os prompts.

In [2]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from src.models import get_llm, token_usage
from src.rag.ingestion import load_file, split_documents
from src.rag.vectorstore import FAISS, build_index, stored_chunks

llm = get_llm()

DOCUMENTS_DIR = Path(settings.documents_dir)
POLICIES_DIR = DOCUMENTS_DIR / "policies"
EXPECTED_CHUNKS = 88  # o que split_documents produz nas 8 políticas com CHUNK_SIZE=1000 e CHUNK_OVERLAP=150

TEST_QUESTIONS = {
    "exchange": "Em quantos dias posso trocar um produto?",
    "shipping": "Qual o valor mínimo da compra pra ter frete grátis?",  # q01 do conjunto de avaliação
    "no_answer": "Vocês vendem garantia estendida? Quanto custa?",  # q27 (no_answer)
}
EXPECTED_SOURCES = {"exchange": "policies/exchange.md", "shipping": "policies/shipping.md"}

NO_ANSWER = "Não encontrei essa informação nas políticas da TechNova."
SYSTEM_PROMPT = (
    "Você é o assistente de atendimento da TechNova, uma loja online de eletrônicos. "
    "Responda à pergunta do cliente usando APENAS os trechos das políticas abaixo, sem conhecimento próprio. "
    f'Se os trechos não trouxerem a resposta, responda exatamente: "{NO_ANSWER}" '
    "Responda em português, em até 3 frases, e indique o número do trecho usado entre colchetes, como [1]."
    "\n\nTRECHOS:\n{context}"
)
PLAIN_SYSTEM_PROMPT = (
    "Você é o assistente de atendimento da TechNova, uma loja online de eletrônicos. "
    "Responda em português, em até 3 frases."
)

print("Políticas:", sorted(p.name for p in POLICIES_DIR.glob("*.md")))
print(f"CHUNK_SIZE={settings.chunk_size} | CHUNK_OVERLAP={settings.chunk_overlap}")

2026-09-27 21:03:17 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini


Políticas: ['cancellation.md', 'exchange.md', 'payment.md', 'privacy.md', 'refund.md', 'shipping.md', 'shipping_2024.md', 'warranty.md']
CHUNK_SIZE=1000 | CHUNK_OVERLAP=150


## Atividade 1: carregar as políticas

Crie `documents`, a lista com os `Document` das 8 políticas, carregados com `load_file`. Os `source` devem ficar
relativos a `data/documents` (ex.: `"policies/exchange.md"`), igual ao índice principal do projeto.

<details><summary>Dica</summary>

`load_file(path, base=DOCUMENTS_DIR)` devolve uma **lista** (um Markdown vira um único `Document`). O argumento `base`
define a pasta a partir da qual o `source` é calculado. Percorra `sorted(POLICIES_DIR.glob("*.md"))`.
</details>

In [3]:
documents = [doc for path in sorted(POLICIES_DIR.glob("*.md")) for doc in load_file(path, base=DOCUMENTS_DIR)]

In [4]:
# Verificação (sem chamadas a modelos)
expected_sources = {f"policies/{p.name}" for p in POLICIES_DIR.glob("*.md")}
assert isinstance(documents, list) and len(documents) == 8, "Esperado uma lista com 8 Documents"
sources = {d.metadata.get("source") for d in documents}
assert sources == expected_sources, f"Confira o base do load_file: os sources vieram como {sorted(sources)[:2]}..."
by_source = {d.metadata["source"]: d for d in documents}
assert by_source["policies/shipping_2024.md"].metadata["status"] == "archived", "shipping_2024.md deveria estar 'archived'"
assert not any(d.page_content.startswith("---") for d in documents), "O front matter não deveria estar no texto"
print("✅ 8 políticas carregadas, com o front matter nos metadados")
for doc in documents:
    print(f"   {doc.metadata['source']:<28} {doc.metadata['status']:<9} {doc.metadata['title']}")

✅ 8 políticas carregadas, com o front matter nos metadados
   policies/cancellation.md     active    Política de Cancelamento de Pedidos
   policies/exchange.md         active    Política de Troca
   policies/payment.md          active    Política de Pagamento
   policies/privacy.md          active    Política de Privacidade e Proteção de Dados
   policies/refund.md           active    Política de Reembolso
   policies/shipping.md         active    Política de Frete e Entrega
   policies/shipping_2024.md    archived  Política de Frete e Entrega
   policies/warranty.md         active    Política de Garantia


## Atividade 2: dividir em trechos

Crie `chunks` com `split_documents`, usando a configuração padrão do projeto (`CHUNK_SIZE` e `CHUNK_OVERLAP` do
`.env`). Lembre o que ele faz: divide o Markdown pelos títulos `##` e `###`, corta os blocos grandes, coloca o
cabeçalho de contexto `[título > seção]` no início de cada trecho e gera um `id` estável.

In [5]:
chunks = split_documents(documents)

2026-09-27 21:03:17 | INFO     | src.rag.ingestion | Documentos divididos | documentos=8 | trechos=88


In [6]:
# Verificação (sem chamadas a modelos)
from collections import Counter

assert all("id" in c.metadata for c in chunks), "Cada trecho deveria ter metadata['id'] (use split_documents)"
assert all(c.page_content.startswith("[") for c in chunks), "Os trechos deveriam começar com o cabeçalho [título > seção]"
assert len(chunks) == EXPECTED_CHUNKS, (
    f"Esperado {EXPECTED_CHUNKS} trechos, veio {len(chunks)}. Use split_documents(documents) sem mudar os parâmetros "
    "e confira CHUNK_SIZE=1000 e CHUNK_OVERLAP=150 no .env."
)
print(f"✅ {len(chunks)} trechos com id, seção e cabeçalho de contexto")
print("   por política:", dict(Counter(c.metadata["source"].split("/")[1] for c in chunks)))
print("\nExemplo de trecho:\n" + chunks[1].page_content[:300], "...")

✅ 88 trechos com id, seção e cabeçalho de contexto
   por política: {'cancellation.md': 10, 'exchange.md': 11, 'payment.md': 11, 'privacy.md': 14, 'refund.md': 9, 'shipping.md': 10, 'shipping_2024.md': 13, 'warranty.md': 10}

Exemplo de trecho:
[Política de Cancelamento de Pedidos > Regra principal]
## Regra principal  
- **Pedidos ainda não enviados podem ser cancelados sem custo.** Isso vale para pedidos com status **processando**, ou seja, com pagamento confirmado mas ainda não despachados.
- **Depois do envio, o pedido não pode mais se ...


## Atividade 3: índice FAISS em memória

Crie `index` com `build_index`. Não salve nada em disco: o índice do exercício fica só em memória, e o índice
principal (`data/faiss_index`) não é tocado.

In [7]:
index = build_index(chunks)  # embeddings dos 88 trechos, em lote

2026-09-27 21:03:17 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=88 | embeddings=azure · text-embedding-3-large


2026-09-27 21:03:17 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


In [8]:
# Verificação (sem novas chamadas)
assert isinstance(index, FAISS), "index deve ser o FAISS devolvido por build_index"
assert index.index.ntotal == EXPECTED_CHUNKS, f"O índice tem {index.index.ntotal} vetores; esperado {EXPECTED_CHUNKS}"
assert set(stored_chunks(index)) == {c.metadata["id"] for c in chunks}, "Os ids do índice deveriam ser os ids dos trechos"
print(f"✅ Índice com {index.index.ntotal} trechos de {index.index.d} dimensões, com os ids estáveis dos trechos")

✅ Índice com 88 trechos de 3072 dimensões, com os ids estáveis dos trechos


## Atividade 4: buscar os 3 trechos mais próximos

Complete `search(question, k=3)`, que devolve os `k` trechos (`Document`) mais próximos da pergunta, do mais próximo
para o menos próximo.

In [9]:
def search(question: str, k: int = 3) -> list[Document]:
    """Os k trechos mais próximos da pergunta."""
    return index.similarity_search(question, k=k)

In [10]:
# Verificação (uma chamada de embeddings por pergunta; nenhuma ao LLM)
for name, question in TEST_QUESTIONS.items():
    found = search(question)
    assert len(found) == 3 and all(isinstance(d, Document) for d in found), "search deve devolver 3 Documents"
    print(question)
    for doc in found:
        print(f"   {doc.metadata['source']:<28} {doc.metadata['status']:<9} {doc.metadata.get('section', '')}")
    if name in EXPECTED_SOURCES:
        assert EXPECTED_SOURCES[name] in [d.metadata["source"] for d in found], (
            f"{EXPECTED_SOURCES[name]} deveria estar entre os 3 primeiros"
        )
        print(f"   ✅ {EXPECTED_SOURCES[name]} está entre os 3 primeiros")

Em quantos dias posso trocar um produto?
   policies/exchange.md         active    Resumo da regra
   policies/exchange.md         active    Direito de arrependimento e prazo ampliado
   policies/exchange.md         active    Troca por outro modelo ou outra cor
   ✅ policies/exchange.md está entre os 3 primeiros


Qual o valor mínimo da compra pra ter frete grátis?
   policies/shipping_2024.md    archived  Frete grátis
   policies/shipping_2024.md    archived  Dúvidas frequentes
   policies/shipping.md         active    Frete grátis
   ✅ policies/shipping.md está entre os 3 primeiros


Vocês vendem garantia estendida? Quanto custa?
   policies/warranty.md         active    
   policies/warranty.md         active    O que a garantia não cobre
   policies/warranty.md         active    O que a garantia cobre


**Observe (para discussão):** na pergunta de frete, quais trechos vieram antes de `policies/shipping.md`? Olhe o
`status` deles. Guarde essa observação para a atividade 5 e para o desafio.

## Atividade 5: prompt com contexto e resposta

Agora a fase de consulta. Complete três peças:

1. `format_context(docs)`: transforma os trechos em um texto numerado, um por bloco, no formato
   `[1] (policies/exchange.md) texto do trecho`, separados por uma linha em branco;
2. `rag_chain`: uma `ChatPromptTemplate` com a mensagem de sistema `SYSTEM_PROMPT` (ela já tem a variável
   `{context}`) e a mensagem do usuário com a variável `{question}`, encadeada ao `llm` com LCEL;
3. `answer_question(question)`: busca os 3 trechos, chama `rag_chain` e devolve um dicionário com `answer` (texto),
   `sources` (lista de `source` dos trechos) e `usage` (`token_usage` da resposta).

In [11]:
def format_context(docs: list[Document]) -> str:
    """Trechos numerados, prontos para o prompt."""
    return "\n\n".join(
        f"[{number}] ({doc.metadata['source']}) {doc.page_content}" for number, doc in enumerate(docs, start=1)
    )


RAG_PROMPT = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("human", "{question}")])
rag_chain = RAG_PROMPT | llm


def answer_question(question: str) -> dict:
    """Busca, monta o prompt e responde. Devolve answer, sources e usage."""
    docs = search(question)
    response = rag_chain.invoke({"context": format_context(docs), "question": question})
    return {
        "answer": response.content,
        "sources": [doc.metadata["source"] for doc in docs],
        "usage": token_usage(response),
    }

In [12]:
# Verificação das peças (sem chamadas a modelos)
fake_docs = [
    Document(page_content="Texto A.", metadata={"source": "policies/a.md"}),
    Document(page_content="Texto B.", metadata={"source": "policies/b.md"}),
]
context = format_context(fake_docs)
assert context.startswith("[1] (policies/a.md) Texto A."), f"Início inesperado: {context[:40]!r}"
assert "\n\n[2] (policies/b.md) Texto B." in context, "O 2o trecho deve vir depois de uma linha em branco: [2] (source) texto"
print("✅ format_context numera os trechos com a fonte")

assert isinstance(RAG_PROMPT, ChatPromptTemplate), "RAG_PROMPT deve ser uma ChatPromptTemplate"
assert set(RAG_PROMPT.input_variables) == {"context", "question"}, f"Variáveis do prompt: {RAG_PROMPT.input_variables}"
assert getattr(rag_chain, "last", None) is llm, "rag_chain deve terminar no llm (RAG_PROMPT | llm)"
print("✅ RAG_PROMPT tem {context} e {question}, e rag_chain termina no llm")

✅ format_context numera os trechos com a fonte
✅ RAG_PROMPT tem {context} e {question}, e rag_chain termina no llm


Agora, as três perguntas de teste (3 chamadas ao LLM):

In [13]:
rag_results = {name: answer_question(question) for name, question in TEST_QUESTIONS.items()}

for name, result in rag_results.items():
    print(f"PERGUNTA: {TEST_QUESTIONS[name]}")
    print(f"FONTES:   {result['sources']}")
    print(f"RESPOSTA: {result['answer']}")
    print(f"TOKENS:   {result['usage']}\n")

PERGUNTA: Em quantos dias posso trocar um produto?
FONTES:   ['policies/exchange.md', 'policies/exchange.md', 'policies/exchange.md']
RESPOSTA: Você pode solicitar a troca em até **30 dias corridos após a entrega** do pedido. Nos **7 primeiros dias**, a desistência pode ser convertida em devolução com reembolso integral, incluindo o frete pago. [1][2]
TOKENS:   {'input_tokens': 574, 'output_tokens': 54, 'total_tokens': 628, 'calls': 1}

PERGUNTA: Qual o valor mínimo da compra pra ter frete grátis?
FONTES:   ['policies/shipping_2024.md', 'policies/shipping_2024.md', 'policies/shipping.md']
RESPOSTA: Frete grátis para compras acima de R$ 199,00, em todo o Brasil. O valor considerado é o total dos produtos no carrinho, antes da aplicação de cupons de desconto. [1]
TOKENS:   {'input_tokens': 539, 'output_tokens': 46, 'total_tokens': 585, 'calls': 1}

PERGUNTA: Vocês vendem garantia estendida? Quanto custa?
FONTES:   ['policies/warranty.md', 'policies/warranty.md', 'policies/warranty.md']
R

In [14]:
# Verificação das respostas
exchange_answer = rag_results["exchange"]["answer"]
assert "30" in exchange_answer, f"A resposta de troca deveria citar 30 dias: {exchange_answer!r}"
print("✅ Troca: a resposta cita o prazo de 30 dias")

no_answer = rag_results["no_answer"]["answer"]
assert "não encontrei" in no_answer.lower(), (
    "Para a pergunta sem resposta na base, o modelo deveria usar a frase de NO_ANSWER. "
    "Confira se o SYSTEM_PROMPT foi usado como mensagem de sistema."
)
assert "R$" not in no_answer, "A resposta sem base não deveria trazer valores"
print("✅ Sem resposta na base: o assistente admitiu que não encontrou, sem inventar preço")

shipping_answer = rag_results["shipping"]["answer"]
if "199" in shipping_answer:
    print("⚠️ Frete: a resposta usou a política ARQUIVADA (R$ 199). Veja o desafio opcional.")
elif "299" in shipping_answer:
    print("✅ Frete: a resposta cita a regra atual (R$ 299)")
else:
    print("ℹ️ Frete: confira a resposta acima e compare com policies/shipping.md (R$ 299,00)")

✅ Troca: a resposta cita o prazo de 30 dias
✅ Sem resposta na base: o assistente admitiu que não encontrou, sem inventar preço
⚠️ Frete: a resposta usou a política ARQUIVADA (R$ 199). Veja o desafio opcional.


## Atividade 6: comparar com a resposta sem contexto

Faça as mesmas três perguntas ao `llm` **sem** trechos, usando `PLAIN_SYSTEM_PROMPT` como mensagem de sistema. Use
`llm.batch` para mandar as três de uma vez e guarde as respostas (`AIMessage`) em `plain_answers`, na mesma ordem de
`TEST_QUESTIONS`.

In [15]:
plain_answers = llm.batch(
    [[("system", PLAIN_SYSTEM_PROMPT), ("human", question)] for question in TEST_QUESTIONS.values()],
    config={"max_concurrency": 3},
)

In [16]:
# Verificação e comparação (sem novas chamadas)
assert isinstance(plain_answers, list) and len(plain_answers) == 3, "plain_answers deve ter 3 respostas"
assert all(hasattr(a, "content") for a in plain_answers), "Cada item deve ser a AIMessage devolvida pelo llm"

for (name, question), plain in zip(TEST_QUESTIONS.items(), plain_answers):
    print(f"PERGUNTA: {question}")
    print(f"   SEM CONTEXTO: {plain.content}")
    print(f"   COM RAG:      {rag_results[name]['answer']}\n")

rag_input = sum(r["usage"]["input_tokens"] for r in rag_results.values())
plain_input = sum(token_usage(a)["input_tokens"] for a in plain_answers)
print(f"Tokens de entrada: sem contexto = {plain_input} | com RAG = {rag_input}")
print("✅ Comparação feita para as 3 perguntas")

PERGUNTA: Em quantos dias posso trocar um produto?
   SEM CONTEXTO: Você pode solicitar a troca em até 7 dias corridos após o recebimento do produto. Se quiser, posso te informar também as condições para a troca.
   COM RAG:      Você pode solicitar a troca em até **30 dias corridos após a entrega** do pedido. Nos **7 primeiros dias**, a desistência pode ser convertida em devolução com reembolso integral, incluindo o frete pago. [1][2]

PERGUNTA: Qual o valor mínimo da compra pra ter frete grátis?
   SEM CONTEXTO: O frete grátis na TechNova é válido para compras acima de R$ 299,00. Se quiser, posso te ajudar a calcular se o seu carrinho já atingiu esse valor.
   COM RAG:      Frete grátis para compras acima de R$ 199,00, em todo o Brasil. O valor considerado é o total dos produtos no carrinho, antes da aplicação de cupons de desconto. [1]

PERGUNTA: Vocês vendem garantia estendida? Quanto custa?
   SEM CONTEXTO: Sim, oferecemos garantia estendida para alguns produtos. O valor varia con

**Observe e responda (para discussão):**

1. Nas perguntas de troca e de garantia estendida, o que a resposta sem contexto disse? Um cliente conseguiria saber
   que ela não vem da TechNova?
2. Quanto custou, em tokens de entrada, colar 3 trechos no prompt? Compare com colar a política inteira (aula 01.1).
3. Na pergunta de frete, a resposta com RAG usou qual valor? Volte às fontes da atividade 4 e explique por quê.

## Desafio opcional: filtrar a política arquivada

A política `shipping_2024.md` está `archived`, mas continua no índice. Mostre a diferença que um filtro faz na
pergunta de frete:

1. busque os 3 trechos da pergunta de frete **com** `filter={"status": "active"}` e compare as fontes com a busca sem
   filtro;
2. responda à pergunta de frete com os trechos filtrados (reutilize `format_context` e `rag_chain`);
3. compare as duas respostas. O que mudou? Por que o modelo não conseguiria, sozinho, saber qual política vale?

<details><summary>Dica</summary>

`index.similarity_search(pergunta, k=3, filter={"status": "active"})`. Repare também que as duas políticas de frete
têm o **mesmo título**; só o metadado `status` as diferencia.
</details>

In [17]:
shipping_question = TEST_QUESTIONS["shipping"]

unfiltered = search(shipping_question)
active_only = index.similarity_search(shipping_question, k=3, filter={"status": "active"})
print("Sem filtro:  ", [(d.metadata["source"], d.metadata["status"]) for d in unfiltered])
print("Só 'active': ", [(d.metadata["source"], d.metadata["status"]) for d in active_only])

filtered_response = rag_chain.invoke({"context": format_context(active_only), "question": shipping_question})
print("\nRESPOSTA SEM FILTRO:", rag_results["shipping"]["answer"])
print("RESPOSTA COM FILTRO:", filtered_response.content)

assert all(d.metadata["status"] == "active" for d in active_only)
assert "299" in filtered_response.content, "Com o filtro, a resposta deveria citar R$ 299,00"
print("\n✅ Com o filtro, só trechos em vigor entram no prompt e a resposta traz a regra atual")

Sem filtro:   [('policies/shipping_2024.md', 'archived'), ('policies/shipping_2024.md', 'archived'), ('policies/shipping.md', 'active')]
Só 'active':  [('policies/shipping.md', 'active'), ('policies/shipping.md', 'active'), ('policies/cancellation.md', 'active')]



RESPOSTA SEM FILTRO: Frete grátis para compras acima de R$ 199,00, em todo o Brasil. O valor considerado é o total dos produtos no carrinho, antes da aplicação de cupons de desconto. [1]
RESPOSTA COM FILTRO: O frete grátis vale para compras acima de R$ 299,00 na modalidade Normal, considerando o total dos produtos após descontos e sem contar o frete. [1]

✅ Com o filtro, só trechos em vigor entram no prompt e a resposta traz a regra atual


**Observe:** na execução acima, sem filtro, os dois primeiros trechos vinham da política **arquivada**, e o modelo
fez exatamente o que o prompt mandou: respondeu com base nos trechos e citou R$ 199 com a referência [1]. A resposta
parece **fundamentada** e está errada. O modelo não falhou; a **busca** falhou, entregando a regra antiga. Com o
filtro, só trechos em vigor chegam ao prompt e a resposta traz R$ 299.

Compare com a atividade 6: sem contexto, o modelo "acertou" os R$ 299 por sorte. Acertar sem fonte não é confiável, e
errar com a fonte errada é ainda pior, porque a citação faz a resposta parecer verificada.

> 💡 **Dica:** experimente trocar `k=3` por `k=5`, com e sem filtro. Mais trechos **não** resolvem o problema da
> política arquivada: ela continua lá, competindo. O que resolve é tirá-la da busca. No módulo 02, o projeto faz isso
> sempre, com `make_filter` e `CUSTOMERS_ONLY`, que também removem o documento interno.

## Resumo

- Um RAG mínimo tem as duas fases: **ingestão** (carregar, dividir, indexar) e **consulta** (buscar, montar o prompt,
  responder). Com o produto cuidando da ingestão, a consulta cabe em poucas linhas.
- Os metadados (`source`, `title`, `section`, `status`) viajam com cada trecho: servem para citar a fonte e para
  filtrar. O cabeçalho `[título > seção]` ajuda a busca e o modelo a entender de onde veio o trecho.
- O prompt restringe o modelo ao contexto e define uma saída segura ("não encontrei"). Sem contexto, o modelo responde
  com a mesma confiança, certo ou errado.
- A busca vetorial não sabe o que está **em vigor**: sem filtro, a política arquivada compete com a atual. Filtrar por
  metadados é parte da recuperação, não um detalhe.
- O que ainda falta, e vem no módulo 02: filtros padronizados (só `active` e só `customers`), busca híbrida para
  termos exatos, reordenação e respostas estruturadas com citações.

## Próxima aula

No módulo 02 começamos pela recuperação de verdade: na aula 02.1 (`02_01_retrievers_and_filters`) vamos transformar a
busca em um *retriever* do LangChain e aplicar sempre os filtros de status e de público com `make_filter` e
`CUSTOMERS_ONLY` ([src/rag/retrieval.py](../src/rag/retrieval.py)).